# Network Update 25_1

In [1]:
import geopandas as gpd
import folium


def show_geom_interactive(
    gdf: gpd.GeoDataFrame,
    *,
    color_col: str | None = "geom_type",
    tiles: str = "CartoDB positron",
    zoom_start: int = 12,
    point_radius: int = 6,
    line_weight: int = 4,
    opacity: float = 0.9,
):
    if gdf is None or gdf.empty:
        raise ValueError("GeoDataFrame is empty.")
    if gdf.crs is None:
        raise ValueError("GeoDataFrame has no CRS.")

    gdf = gdf.to_crs(4326).copy()
    geom_col = gdf.geometry.name

    bounds = gdf.total_bounds
    center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

    m = folium.Map(location=center, zoom_start=zoom_start, tiles=tiles)

    tooltip_cols = [c for c in gdf.columns if c != geom_col]

    color_map = {
        "old": "#ff2d2d",
        "new": "#2d6cff",
    }

    def get_color(props):
        if color_col and color_col in props:
            return color_map.get(props[color_col], "#3388ff")
        return "#3388ff"

    point_mask = gdf.geom_type.isin(["Point", "MultiPoint"])
    gdf_points = gdf[point_mask]
    gdf_other = gdf[~point_mask]

    if not gdf_other.empty:
        folium.GeoJson(
            gdf_other,
            style_function=lambda f: {
                "color": get_color(f["properties"]),
                "weight": line_weight,
                "opacity": opacity,
                "fillOpacity": 0.4,
            },
            tooltip=folium.GeoJsonTooltip(fields=tooltip_cols),
        ).add_to(m)

    if not gdf_points.empty:
        for _, row in gdf_points.iterrows():
            geom = row[geom_col]

            if geom.geom_type == "Point":
                pts = [geom]
            elif geom.geom_type == "MultiPoint":
                pts = list(geom.geoms)
            else:
                pts = []

            tooltip_text = "<br>".join(
                f"<b>{c}</b>: {row[c]}" for c in tooltip_cols
            )

            color = color_map.get(row.get(color_col), "#3388ff")

            for pt in pts:
                folium.CircleMarker(
                    location=[pt.y, pt.x],
                    radius=point_radius,
                    color=color,
                    fill=True,
                    fill_color=color,
                    fill_opacity=opacity,
                    tooltip=folium.Tooltip(tooltip_text),
                ).add_to(m)

    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    return m

## Whats new? 
Its that time of the year again, lets see how many things needs to be changed this time.

In [12]:
import geopandas as gpd
import folium
from psycopg2 import connect
from psycopg2 import sql
from psycopg2.extras import execute_values
from pathlib import Path
import pandas as pd
import pandas.io.sql as pandasql
import configparser
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
CONFIG = configparser.ConfigParser()
CONFIG.read(str(Path.home().joinpath('db.cfg')))
dbset = CONFIG['DBSETTINGS']
con = connect(**dbset)
from sqlalchemy import text
connection_engine = create_engine(
    URL.create(
        drivername = "postgresql",
        host = CONFIG['DBSETTINGS']['host'],
        database = CONFIG['DBSETTINGS']['database'],
        username = CONFIG['DBSETTINGS']['user'],
        password = CONFIG['DBSETTINGS']['password']))
cur = connection_engine.raw_connection().cursor()

### Number of nodes that needs to be updated

0!

In [29]:
sql = '''
select node_id::bigint from congestion.congestion_nodes_24_4
except 
select node_id::bigint  from here.routing_nodes_25_1
    '''
with con : 
    nodes = pandasql.read_sql(sql, connection_engine) 
print(nodes)   

Empty DataFrame
Columns: [node_id]
Index: []


### How about geom for the nodes changes?

Literally all geom changed.....

In [30]:
sql = '''
with temp as (
select node_id::bigint, geom from congestion.congestion_nodes_24_4 
except 
select node_id::bigint, geom  from here.routing_nodes_25_1)

select (select count(1) from temp), count(1)
from congestion.congestion_nodes_24_4 

    '''
with con : 
    nodes = pandasql.read_sql(sql, connection_engine) 
print(nodes)   

   count  count
0     19   3111


19 nodes with moved distance, 27 being the biggest shift.

In [31]:
sql = '''
with temp as (
select node_id::bigint, geom from congestion.congestion_nodes_24_4 
except 
select node_id::bigint, geom  from here.routing_nodes_25_1)


SELECT distinct node_id, 
ST_distance(ST_Transform(temp.geom, 2952), ST_Transform(r.geom, 2952))::double precision 
FROM temp 
INNER JOIN here.routing_nodes_25_1 r USING (node_id)
where ST_distance(ST_Transform(temp.geom, 2952), ST_Transform(r.geom, 2952))::double precision >0
order by st_distance desc

    '''
with con : 
    nodes = pandasql.read_sql(sql, connection_engine) 
print(nodes)   

        node_id  st_distance
0   886126206.0    27.412367
1    30347302.0    21.439269
2    30366718.0    14.397555
3    30352281.0     9.132587
4    30342212.0     8.144147
5    30451381.0     7.824355
6    30334865.0     7.791419
7    30420332.0     7.263280
8    30443224.0     7.249116
9    30419055.0     7.090907
10   30487696.0     6.419876
11   30352375.0     5.612884
12   30414720.0     4.516384
13   30362739.0     4.443775
14   30478435.0     3.428661
15   30414994.0     3.412068
16   30414999.0     3.412028
17   30352282.0     2.659062
18   30451390.0     1.956048


Doesn't look that drastic

In [4]:
sql = """
WITH temp AS (
    SELECT node_id::bigint, geom
    FROM congestion.congestion_nodes_24_4
    EXCEPT
    SELECT node_id::bigint, geom
    FROM here.routing_nodes_25_1
),
moved AS (
    SELECT
        temp.node_id,
        ST_Distance(ST_Transform(temp.geom, 2952),ST_Transform(r.geom, 2952)
        )::double precision AS st_distance,
        temp.geom AS old_geom,
        r.geom AS new_geom
    FROM temp
    JOIN here.routing_nodes_25_1 r USING (node_id)
    WHERE ST_Distance(ST_Transform(temp.geom, 2952),ST_Transform(r.geom, 2952)) > 0
)

SELECT node_id,st_distance,'old' AS geom_type,old_geom AS geom
FROM moved
UNION ALL
SELECT node_id, st_distance, 'new' AS geom_type, new_geom AS geom
FROM moved

ORDER BY st_distance DESC;
"""
points = gpd.GeoDataFrame.from_postgis(sql, con, geom_col='geom')
points = points.to_crs('epsg:4326')

/data/jupyterhub/.venv/lib/python3.10/site-packages/geopandas/io/sql.py:170: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [6]:
m = show_geom_interactive(points)
m

### Create a new nodes table based on the new geometry

In [13]:
sql = '''
    CREATE TABLE congestion.temp_congestion_nodes_25_1 AS 
    SELECT distinct nodes.node_id::bigint, nodes.geom, '25_1' as ver_id
    FROM here.routing_nodes_25_1 nodes
	INNER JOIN congestion.congestion_nodes_24_4 USING (node_id);

    ALTER TABLE congestion.temp_congestion_nodes_25_1 OWNER TO congestion_admins;
    '''
with connection_engine.begin() as con:
    con.execute(text(sql))

## Number of links that needs to be updated

61 links

In [32]:
sql = '''
select link_dir from congestion.congestion_links_24_4
except 
select link_dir from here.routing_streets_25_1
    '''
with con : 
    links = pandasql.read_sql(sql, connection_engine) 
print(links)   

       link_dir
0     29580631T
1     29583402F
2    810691130T
3     29571496T
4   1376895389T
..          ...
56  1258684691F
57   783557165T
58    29571496F
59    29580631F
60  1258684691T

[61 rows x 1 columns]


## Length differences

140 links changes, biggest changes around 30 m changes. Links with >20m changes are all on highways.

In [19]:
sql = '''
select link_dir, 
    ST_length(ST_transform(retired_nodes.geom, 2952)) - ST_length(ST_transform(z.geom, 2952))::int, z.geom
    
from congestion.congestion_links_24_4 AS retired_nodes
inner join here.routing_streets_25_1 z using (link_dir) 
where ST_length(ST_transform(retired_nodes.geom, 2952)) - ST_length(ST_transform(z.geom, 2952))  >0
order by abs(ST_length(ST_transform(retired_nodes.geom, 2952)) - ST_length(ST_transform(z.geom, 2952)) ) desc
    '''
with con : 
    links = pandasql.read_sql(sql, connection_engine) 
print(links)   

        link_dir   ?column?                                               geom
0     133788590F  30.851717  0102000020E61000000A000000A1A17F828BD553C0C4CE...
1     832667674T  28.098279  0102000020E61000000300000044696FF085DF53C0556A...
2     133790141F  23.584754  0102000020E6100000040000008BFD65F7E4E153C0D53E...
3      29576576T  21.272439  0102000020E61000000200000059FAD005F5DF53C00938...
4      29576576F  21.272439  0102000020E610000002000000CEC29E76F8DF53C05709...
..           ...        ...                                                ...
135  1114553813F  -0.071917  0102000020E61000000200000090DAC4C9FDDA53C0FA7E...
136  1327981227F   0.070239  0102000020E6100000020000002041F163CCD153C0B62D...
137  1327981227T   0.070239  0102000020E610000002000000AF997CB3CDD153C05A12...
138  1299499655T  -0.058107  0102000020E6100000020000002041F163CCD153C0B62D...
139  1299499655F  -0.058107  0102000020E610000002000000FD6A0E10CCD153C0BD52...

[140 rows x 3 columns]


In [24]:
sql = '''
select link_dir, 
    ST_length(ST_transform(retired_nodes.geom, 2952)) - ST_length(ST_transform(z.geom, 2952))::int, z.geom
    
from congestion.congestion_links_24_4 AS retired_nodes
inner join here.routing_streets_25_1 z using (link_dir) 
where ST_length(ST_transform(retired_nodes.geom, 2952)) - ST_length(ST_transform(z.geom, 2952))  > 20
order by abs(ST_length(ST_transform(retired_nodes.geom, 2952)) - ST_length(ST_transform(z.geom, 2952)) ) desc
    '''
with con : 
    links = pandasql.read_sql(sql, connection_engine) 
print(links)   

       link_dir   ?column?                                               geom
0    133788590F  30.851717  0102000020E61000000A000000A1A17F828BD553C0C4CE...
1    832667674T  28.098279  0102000020E61000000300000044696FF085DF53C0556A...
2    133790141F  23.584754  0102000020E6100000040000008BFD65F7E4E153C0D53E...
3     29576576F  21.272439  0102000020E610000002000000CEC29E76F8DF53C05709...
4     29576576T  21.272439  0102000020E61000000200000059FAD005F5DF53C00938...
5    946302229F  20.825794  0102000020E61000000200000035D252793BDC53C0B79C...
6    946302229T  20.825794  0102000020E610000002000000C74B378941DC53C0EFFE...
7    993248194F  21.382276  0102000020E61000000600000012312592E8DF53C09F59...
8    755273743T  20.861013  0102000020E61000000200000084F57F0EF3E153C0E3C7...
9   1347772786F  20.420814  0102000020E610000002000000FBCBEEC9C3D653C0B8CC...
10  1347772786T  20.420814  0102000020E610000002000000A2B437F8C2D653C0FF78...
11  1347955689T  20.734137  0102000020E6100000050000005A2A6F4738

In [25]:
with connection_engine.connect() as con:
    links = gpd.read_postgis(sql, con, geom_col="geom")
links = links.to_crs('epsg:4326')

In [26]:
m = show_geom_interactive(links)
m

## Create links
The function create new congestion links, by replacing all link's geometry to the new version. Delete segments that was affected by retired links, and reroute those missing segments.

In [ ]:
sql = '''
    SELECT congestion.rebuild_temp_links('24_4','25_1');
    '''
with connection_engine.begin() as con:
    con.execute(text(sql))

Check to see if all segment_ids are inserted. They might not be correct, but we will validate and update in later steps.

In [33]:
sql = '''
    select distinct segment_id from congestion.congestion_links_24_4
    except
    select distinct segment_id from congestion.temp_network_links_25_1
    '''
with con : 
    links = pandasql.read_sql(sql, connection_engine) 
print(links)   

Empty DataFrame
Columns: [segment_id]
Index: []


## Create segments

In [ ]:
sql = '''
    select congestion.rebuild_temp_segments('25_1')
    '''
with connection_engine.begin() as con:
    con.execute(text(sql))

## Check and see if there are new traffic signals we can add

Check with the layer `congestion.excluded_signals` which contains all signals that are currently not included in the network, and compare with the `gis.traffic_signal` to find if there are new traffic signal added. 

First join to intersection_id using nearest neighbour, and then join to routing nodes using STD_within, because there are often more than 1 node_id corresponding to any intersection_id or px, due to the natural of here nodes.

This excludes removed signals and temporary signals.

```sql
SELECT congestion.create_temp_int_px_nodes('25_1')
```
This above query creates the table `congestion.temp_int_px_nodes_ver_id`, in this case `congestion.temp_int_px_nodes_25_1`.

### Manual work time

Open QGIS, and display the newly created layer `congestion.temp_int_px_nodes_25_1`. 

Make sure all these traffic signals are legit additions, check for: 

    - are they on the network
    - does centreline starts/ends at that signal
    - is the joined intersection_id correct
    - check if the node_ids are correct as well

You will likely need to update things manually, as nearest neighbour joins are not always perfect. Use the following handy SQLs and record your manual changes. 

```sql
UPDATE TABLE congestion.temp_int_px_nodes_25_1
SET intersection_id = , node_id = 
WHERE px = 

```

Remove incorrectly joined node_id, made sure intersection_id joined to it is correct.

```sql
DELETE FROM congestion.temp_int_px_nodes_25_1
WHERE node_id in (30415071, 1254271925);

DELETE FROM congestion.temp_int_px_nodes_25_1
WHERE px = '2687'; -- not on network
```

For any new px that you decided not to add to the network, insert them into the `congestion.excluded_signals`, so we either 
- reconsider them if they are on the network but there is just no centreline intersection at this point
- ignore them if they are not on the network 

```sql
-- Inserting the px 2687 to the excluded signals table
SELECT * FROM congestion.insert_excluded_px( '2687', '25_1');
```

## Any new centreline intersections at traffic signals ?

We can use the layer `congestion.excluded_signals` to figure out whether there are signals we care about, that has a new intersection located close to it. 

Run the following function to see if there are any different intersections matches.
It will display the px, old_int, new_int, old_dist, new_dist.
Where old_int is the old intersection_id that is the closest to the px
new_int the latest intersection_id in gis_core.centreline_intersection that is closest to the px
old_dist old intersection distance from the px
new_dist new intersection's distance from the px
```sql
SELECT * FROM congestion.check_for_new_intersections_px();
```


If there are any, check and see if they are legit. If so, insert into `congestion.temp_int_px_nodes_25_1`

## Separate links with new nodes added

After adding new traffic signals, now we can separate the links and re-add new segments.

```sql
-- Dry run first 
SELECT * FROM congestion.seperate_segments_w_nodes(
    ver_id := '25_1',
    dry_run := TRUE
);
```

Everything looks ok, then 

```sql
-- Actually run it 
SELECT * FROM congestion.seperate_segments_w_nodes(
    ver_id := '25_1',
    dry_run := False
);
```

## Insert new nodes added to congestion nodes

```sql
INSERT INTO congestion.temp_network_nodes_25_5
SELECT node_id, node_geom, '25_1' as ver_id
FROM congestion.temp_int_px_nodes_25_1
```

## Finalize px int nodes lookup table

In the previous steps we have only added new nodes px int pairs to the `congestion.temp_int_px_nodes_25_1` table. We need to add all the existing nodes lookup to the table as well. Making sure to also update any node_ids that changed, as well as intersection_id and geoms that got updated.

Since I created the last one very recently, there is nothing to update.

In [ ]:
sql = '''
SELECT * FROM congestion.congestion_nodes_lookup_24_4
LEFT JOIN gis_core.centreline_intersection_point_latest a USING(intersection_id)
WHERE a.intersection_id is null
    '''
with con : 
    ints = pandasql.read_sql(sql, connection_engine) 
print(ints)   

but lets say hypothetically there are.... we will have to find the new intersection_id.

Will have to add to temp_int_px_nodes_25_1 table.

## Construct the nodes look up table

The following function recreates congestion_nodes_lookup table for the current version.  

```sql
-- creates congestion.temp_congestion_nodes_lookup_25_1
SELECT congestion.rebuild_temp_nodes_lookup(
    '24_4' , -- older version
	'25_1' -- new version
```

## Centreline Lookup updates

There are 39 segments with centreline_id that is outdated
```sql
SELECT distinct segment_id, from_int, to_int
FROM (SELECT segment_id, from_int, to_int, unnest(centreline_ids) as centreline_id, geom 
		FROM congestion.congestion_centreline_24_4) a
LEFT JOiN gis_core.centreline_latest USING (centreline_id)
WHERE centreline_latest.centreline_id IS NULL
```

There are 48 new segments that needs to be conflated
```sql

WITH new_segments AS (
SELECT segment_id, start_vid, end_vid
FROM congestion.temp_network_segments_25_1
EXCEPT
SELECT  segment_id, start_vid, end_vid 
FROM congestion.congestion_segments_24_4
ORDER BY segment_id)
SELECT segment_id, s.intersection_id as start_vid, e.intersection_id as end_vid
FROM new_segments
left join congestion.temp_congestion_nodes_lookup_25_1 s on start_vid = node_id
left join congestion.temp_congestion_nodes_lookup_25_1 e on end_vid = e.node_id
```

Run the following function with dry run to check all routed results, if they look ok, then insert in to the table

```sql
CREATE TABLE congestion.temp_congestion_centreline_25_5 AS
WITH outdated_segments AS (
SELECT distinct segment_id, from_int, to_int
FROM (SELECT segment_id, from_int, to_int, unnest(centreline_ids) as centreline_id, geom 
        FROM congestion.congestion_centreline_24_4) a
LEFT JOiN gis_core.centreline_latest USING (centreline_id)
WHERE centreline_latest.centreline_id IS NULL)

,  new_segments AS (
SELECT segment_id, start_vid, end_vid
FROM congestion.temp_network_segments_25_1
EXCEPT
SELECT  segment_id, start_vid, end_vid 
FROM congestion.congestion_segments_24_4
ORDER BY segment_id)

, new_segments_w_ints AS 
(SELECT segment_id, s.intersection_id as from_int, e.intersection_id as to_int
FROM new_segments
left join congestion.temp_congestion_nodes_lookup_25_1 s on start_vid = node_id
left join congestion.temp_congestion_nodes_lookup_25_1 e on end_vid = e.node_id)

, need_update AS 
(SELECT * FROM outdated_segments
union 
SELECT * FROM new_segments_w_ints)

, results AS (
    SELECT t.from_int, t.to_int, edge, cost, agg_cost, seq, segment_id, node, path_seq
    FROM need_update t
    CROSS JOIN LATERAL pgr_trsp(
        $$
        SELECT
            id,
            source::int,
            target::int,
            cost_length::int AS cost
        FROM gis_core.routing_centreline_directional_higher_rc
        $$,
		$$SELECT path, cost FROM gis_core.centreline_routing_restrictions_higher_rc $$,
        t.from_int, t.to_int,true
    ) AS route)
	
SELECT d.segment_id, 
		d.from_int, 
		d.to_int, 
		array_agg(r.centreline_id ORDER BY path_seq) AS centreline_ids,
		ST_LineMerge(ST_Union(r.geom)) AS geom
from results d
JOIN gis_core.routing_centreline_directional_higher_rc r ON d.edge = r.id
GROUP BY d.segment_id, d.from_int, d.to_int;
```

```sql
INSERT INTO congestion.temp_congestion_centreline_25_5

WITH temp as (
SELECT 
	distinct segment_id, from_int, to_int
FROM (SELECT segment_id, from_int, to_int, unnest(centreline_ids) as centreline_id, geom 
        FROM congestion.congestion_centreline_24_4) a
LEFT JOIN gis_core.centreline_latest USING (centreline_id)
WHERE centreline_latest.centreline_id IS NULL)

, prep AS (
SELECT con.segment_id, con.from_int, con.to_int, unnest(centreline_ids) AS centreline_id
FROM congestion.congestion_centreline_24_4 con
LEFT JOIN temp USING (segment_id)
WHERE temp.segment_id is null)

SELECT segment_id, from_int, to_int, array_agg(centreline_id) AS centreline_ids,
		ST_LineMerge(ST_Union(geom)) AS geom
from prep
inner join gis_core.centreline_latest USING (centreline_id)
GROUP BY segment_id, from_int, to_int;
```

## Retired Segments